<a href="https://colab.research.google.com/github/Aakashmosa-20/Banking-Knowledge-agent-RAG-OKF/blob/main/Banking_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install -q sentence-transformers
import os, re, glob
import numpy as np
print("ready")


ready


In [3]:
NOTES = {
"knowledge/accounts/savings-account.md": """---
type: FAQ
title: Savings Account
---
# Savings Account
## Purpose
A savings account is used to keep money safely and carry out everyday banking transactions.
## Opening
A bank may ask for an application and identity/address documents. Exact requirements depend on the bank and applicable rules.""",

"knowledge/accounts/current-account.md": """---
type: FAQ
title: Current Account
---
# Current Account
## Purpose
A current account is commonly designed for businesses and frequent transactions.
## Charges
Account fees, minimum-balance rules, and transaction limits vary by bank and account type.""",

"knowledge/cards/debit-card.md": """---
type: Safety
title: Debit Card Safety
---
# Debit Card Safety
## PIN and OTP
Never share your PIN, OTP, CVV, password, or full card details with anyone.
## Lost Card
If a card is lost, contact the bank through its official app, website, or verified customer-care channel to block it.""",

"knowledge/loans/personal-loan.md": """---
type: FAQ
title: Personal Loan
---
# Personal Loan
## Eligibility
Eligibility may depend on income, employment, credit history, existing obligations, and the lender's criteria.
## Interest and Fees
Interest rates, processing fees, repayment periods, and other charges vary. Read the Key Facts Statement and loan agreement before accepting an offer.""",

"knowledge/digital-banking/upi.md": """---
type: Safety
title: UPI Payments
status: review
---
# UPI Payments
## Sending Money
Check the recipient name and amount before approving a payment.
## Collect Requests
Entering a UPI PIN generally authorizes a payment; do not enter it just to receive money. Verify unexpected collect requests before taking action."""
}

for path, text in NOTES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print("saved", path)


saved knowledge/accounts/savings-account.md
saved knowledge/accounts/current-account.md
saved knowledge/cards/debit-card.md
saved knowledge/loans/personal-loan.md
saved knowledge/digital-banking/upi.md


In [4]:
files = [
    f for f in sorted(glob.glob("knowledge/**/*.md", recursive=True))
    if os.path.basename(f) not in ("index.md", "log.md")
]

problems = 0
for f in files:
    with open(f, encoding="utf-8") as handle:
        text = handle.read()
    missing = []
    if "type:" not in text:
        missing.append("a type: line")
    if not re.search(r"^# .+", text, re.M):
        missing.append("a # title")
    if "## " not in text:
        missing.append("at least one ## section")
    if missing:
        problems += 1
        print("PROBLEM", f, "is missing", ", ".join(missing))
    else:
        print("OK", f)

index = "---\\ntype: Index\\ntitle: Banking Knowledge Index\\n---\\n# Index\\n"
index += "".join(f"- {f.replace('knowledge/', '')}\\n" for f in files)
with open("knowledge/index.md", "w", encoding="utf-8") as handle:
    handle.write(index)
print("\\nAll notes are good" if problems == 0 else f"\\n{problems} note(s) need fixing")
print("saved knowledge/index.md")


OK knowledge/accounts/current-account.md
OK knowledge/accounts/savings-account.md
OK knowledge/cards/debit-card.md
OK knowledge/digital-banking/upi.md
OK knowledge/loans/personal-loan.md
\nAll notes are good
saved knowledge/index.md


In [5]:
from sentence_transformers import SentenceTransformer

def load_pieces():
    pieces = []
    for path in sorted(glob.glob("knowledge/**/*.md", recursive=True)):
        if os.path.basename(path) in ("index.md", "log.md"):
            continue
        with open(path, encoding="utf-8") as handle:
            text = handle.read()
        title_match = re.search(r"^# (.+)$", text, re.M)
        if not title_match:
            continue
        title = title_match.group(1)
        for part in text.split("## ")[1:]:
            pieces.append({
                "file": path,
                "text": f"{title}: {part.strip()}",
                "status": "review" if "status: review" in text else ""
            })
    return pieces

PIECES = load_pieces()
model = SentenceTransformer("all-MiniLM-L6-v2")
VECTORS = model.encode([p["text"] for p in PIECES], normalize_embeddings=True)

def find(question):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = VECTORS @ q
    i = int(np.argmax(scores))
    return float(scores[i]), PIECES[i]

print(len(PIECES), "pieces")
score, piece = find("What should I do if my debit card is lost?")
print(round(score, 2), piece["file"])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

10 pieces
0.73 knowledge/cards/debit-card.md


In [6]:
LIMIT = 0.30
USE_MODEL = False  # Keep False for the simple, retrieval-only version.

def agent(question, show=True):
    score, best = find(question)
    if score < LIMIT:
        file, answer = None, "I could not find this in the banking notes."
    else:
        file = best["file"]
        answer = best["text"]
        if best["status"] == "review":
            answer = "REVIEW WARNING: verify this note against current official guidance. " + answer
    if show:
        print("Q:", question)
        print("FOUND IN:", file, f"(score {score:.2f})")
        print("ANSWER:", answer, "\\n")
    return file

agent("What should I do if my debit card is lost?")
agent("What affects personal loan eligibility?")
agent("What is today's exchange rate?")


Q: What should I do if my debit card is lost?
FOUND IN: knowledge/cards/debit-card.md (score 0.73)
ANSWER: Debit Card Safety: Lost Card
If a card is lost, contact the bank through its official app, website, or verified customer-care channel to block it. \n
Q: What affects personal loan eligibility?
FOUND IN: knowledge/loans/personal-loan.md (score 0.80)
ANSWER: Personal Loan: Eligibility
Eligibility may depend on income, employment, credit history, existing obligations, and the lender's criteria. \n
Q: What is today's exchange rate?
FOUND IN: None (score 0.22)
ANSWER: I could not find this in the banking notes. \n


In [7]:
TESTS = [
    ("What is a savings account used for?", "knowledge/accounts/savings-account.md"),
    ("What documents are needed to open an account?", "knowledge/accounts/savings-account.md"),
    ("Who commonly uses a current account?", "knowledge/accounts/current-account.md"),
    ("What should I do if my debit card is lost?", "knowledge/cards/debit-card.md"),
    ("Should I share my OTP?", "knowledge/cards/debit-card.md"),
    ("What affects personal loan eligibility?", "knowledge/loans/personal-loan.md"),
    ("Do loan charges vary by bank?", "knowledge/loans/personal-loan.md"),
    ("What should I check before sending a UPI payment?", "knowledge/digital-banking/upi.md"),
    ("Should I enter a UPI PIN to receive money?", "knowledge/digital-banking/upi.md"),
    ("What is today's exchange rate?", None),
]

score = 0
for q, expected in TESTS:
    got = agent(q, show=False)
    ok = got == expected
    score += ok
    print(("PASS " if ok else "CHECK ") + q +
          ("" if ok else f" (expected {expected}, got {got})"))
print(f"\\nSCORE: {score}/{len(TESTS)}")


PASS What is a savings account used for?
PASS What documents are needed to open an account?
PASS Who commonly uses a current account?
PASS What should I do if my debit card is lost?
PASS Should I share my OTP?
PASS What affects personal loan eligibility?
PASS Do loan charges vary by bank?
PASS What should I check before sending a UPI payment?
PASS Should I enter a UPI PIN to receive money?
PASS What is today's exchange rate?
\nSCORE: 10/10


In [8]:
import shutil
archive_path = shutil.make_archive("banking_knowledge", "zip", "knowledge")
print("Saved:", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except Exception:
    print("Download banking_knowledge.zip from the Files panel.")


Saved: /content/banking_knowledge.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
%pip install -q gradio
import gradio as gr

def ask_agent(question):
    if not question.strip():
        return "", "", "Type a question first."
    score, best = find(question)
    if score < LIMIT:
        return "No matching note", f"{score:.2f}", "I could not find this in the banking notes."
    answer = best["text"]
    if best["status"] == "review":
        answer = "REVIEW WARNING: verify this note against current official guidance. " + answer
    return f'{best["file"]}\\n\\n{best["text"]}', f"{score:.2f}", answer

with gr.Blocks(title="Banking Knowledge Agent") as demo:
    gr.Markdown("# Banking Knowledge Agent\\nAsk a question. The agent searches the saved notes first.")
    q = gr.Textbox(label="Your question", placeholder="What should I do if my debit card is lost?")
    btn = gr.Button("Ask")
    found = gr.Textbox(label="FOUND IN (source note)", lines=4)
    score_box = gr.Textbox(label="Similarity score")
    ans = gr.Textbox(label="ANSWER", lines=4)
    gr.Examples([
        "What is a savings account used for?",
        "What affects personal loan eligibility?",
        "Should I share my OTP?",
        "What should I check before sending a UPI payment?"
    ], inputs=q)
    btn.click(ask_agent, q, [found, score_box, ans])
    q.submit(ask_agent, q, [found, score_box, ans])

demo.launch(share=False, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>